# 🔬 Notebook 01 — Feature Engineering & Feature Store

**Goal:** Join Phase 1 Lakehouse data with credit risk features to build a unified Feature Store.

This Silver-layer table becomes the single source of truth for all credit risk models.

> **Run time:** ~3 min

In [ ]:
# Import PySpark SQL functions used to engineer account-level credit risk features
from pyspark.sql import functions as F

# ── Load Phase 1 tables from Lakehouse ──────────────────────────────────────
# Load account master data and historical loan transactions from the Lakehouse
df_accounts = spark.table('bank_accounts')
df_txns     = spark.table('silver_loan_transactions')

# Validate the source table sizes before building the feature store
print(f'Accounts: {df_accounts.count()}')
print(f'Transactions: {df_txns.count()}')


## Step 1 — Aggregate Transaction Features per Account

In [ ]:
# Aggregate transaction behaviour per account
# Aggregate transaction activity into account-level behavior signals for default modeling
df_txn_features = df_txns.groupBy('AccountID').agg(
    F.count('TransactionID').alias('TotalTransactions'),
    F.round(F.sum('Amount'), 2).alias('TotalLoanAmount'),
    F.round(F.avg('Amount'), 2).alias('AvgTransactionAmount'),
    F.countDistinct('LoanID').alias('NumLoans'),
    F.round(F.max('Amount'), 2).alias('MaxTransactionAmount')
)

# Preview the engineered transaction features to verify the aggregation results
print('Transaction features aggregated:')
df_txn_features.show(5)


## Step 2 — Load Credit Risk Features CSV

In [ ]:
# Load the new credit risk features
# Load the borrower credit risk feature file from the Lakehouse Files area
df_credit = spark.read.option('header','true').option('inferSchema','true') \
    .csv('Files/credit_risk_features.csv')

# Check the credit feature file row count and schema before joining it
print(f'Credit features: {df_credit.count()} rows')
df_credit.printSchema()


## Step 3 — Join All Features into Feature Store

In [ ]:
# Join accounts + transaction features + credit risk features
# Join account attributes, transaction aggregates, and borrower features into one training table
df_feature_store = df_accounts \
    .select('AccountID','AccountType','Branch','Status') \
    .join(df_txn_features, on='AccountID', how='left') \
    .join(df_credit, on='AccountID', how='inner') \
    .fillna({'TotalTransactions': 0, 'TotalLoanAmount': 0.0,
             'AvgTransactionAmount': 0.0, 'NumLoans': 0,
             'MaxTransactionAmount': 0.0})

# Inspect the assembled feature store to confirm the expected columns and row count
print(f'Feature Store rows: {df_feature_store.count()}')
print(f'Features: {len(df_feature_store.columns)}')
df_feature_store.show(5)


## Step 4 — Save Feature Store as Silver Delta Table

In [ ]:
# Persist the engineered credit risk feature store as a Delta table for downstream notebooks
df_feature_store.write.format('delta').mode('overwrite') \
    .saveAsTable('silver_credit_risk_features')

print('✅ Feature Store saved as silver_credit_risk_features')
# Review the default-label balance in the final modeling dataset
print(f'\nClass balance:')
df_feature_store.groupBy('IsDefault').count().show()


## Step 5 — Exploratory Analysis

In [ ]:
%%sql
-- Summarize default rates by credit score band to confirm the expected risk gradient
-- Default rate by credit score band
SELECT
    CASE
        WHEN CreditScore >= 750 THEN 'Excellent (750+)'
        WHEN CreditScore >= 670 THEN 'Good (670-749)'
        WHEN CreditScore >= 580 THEN 'Fair (580-669)'
        ELSE 'Poor (<580)'
    END AS CreditScoreBand,
    COUNT(*) AS TotalAccounts,
    SUM(IsDefault) AS Defaults,
    ROUND(SUM(IsDefault) * 100.0 / COUNT(*), 1) AS DefaultRate_Pct
FROM silver_credit_risk_features
GROUP BY CreditScoreBand
ORDER BY DefaultRate_Pct DESC
